# CFM Agent v5.2.2 Delta-Governed — Consistency Test

Run the delta-governed CFM agent **3 times on the same origin date** and compare:
- The LLM's proposed rank/uncertainty actions per horizon (its qualitative view)
- The evidence tier the policy actually granted (may clip the rank)
- The historical delta percentiles used as the governor (should be IDENTICAL across
  runs — they come from price history, not the LLM, so any difference here would be a bug)
- The final point forecasts and 80% intervals
- The rationale text

Unlike the Scenario Schema consistency tests, CFM's point forecast is never invented
directly by the LLM — it's always `ensemble_point + a Python-computed shift`. So the
interesting question here isn't "does the price drift wildly" (it structurally can't,
by design) but "does the LLM's own rank/evidence-tier assessment stay consistent".

## Setup

In [1]:
import pandas as pd
import numpy as np
from aieng.forecasting.models import LITE_MODEL
from aieng.forecasting.evaluation.prediction import ContinuousForecast
from aieng.forecasting.evaluation.task import ForecastingTask
from energy_oil_forecasting.data import WTI_SERIES_ID, build_wti_multivariate_service
from energy_oil_forecasting.cfm_agent_v_5_2_2_delta_governed import (
    build_cfm_agent_config_delta_governed,
    build_cfm_agent_predictor_delta_governed,
)

# Configuration
# Yesterday's business day, not today: today's WTI close may not be cached yet,
# and CFM (like the scenario_schema tests) doesn't need a resolved outcome to run —
# this just needs to be a date the price cache actually has data through.
ORIGIN_DATE = pd.Timestamp.now().normalize() - pd.tseries.offsets.BDay(1)
NUM_RUNS = 3
HORIZONS = [5, 10, 21]

data_service = build_wti_multivariate_service()
print(f"Testing CFM Agent v5.2.2 Delta-Governed on origin: {ORIGIN_DATE.date()}")
print(f"Horizons: {HORIZONS} business days")
print(f"Number of runs: {NUM_RUNS}")

Testing CFM Agent v5.2.2 Delta-Governed on origin: 2026-08-21
Horizons: [5, 10, 21] business days
Number of runs: 3


## Run 3 Times

In [2]:
results = []

# Task and context are built once — same origin, same information cutoff,
# for every run. No scoring against realized outcomes (this is a consistency
# test, not a backtest).
task = ForecastingTask(
    task_id="wti_forecast",
    target_series_id=WTI_SERIES_ID,
    horizons=HORIZONS,
    frequency="B",
    description="WTI price forecast",
)
context = data_service.context(as_of=ORIGIN_DATE)

for run_num in range(NUM_RUNS):
    print(f"\n{'='*72}")
    print(f"RUN {run_num + 1} / {NUM_RUNS}")
    print(f"{'='*72}")

    # Build a fresh predictor each run so nothing is cached/reused between runs.
    config = build_cfm_agent_config_delta_governed(model=LITE_MODEL)
    predictor = build_cfm_agent_predictor_delta_governed(config)

    predictions = predictor.predict(task, context)

    results.append({
        "run_num": run_num + 1,
        "predictions": predictions,
    })

    for pred in predictions:
        if isinstance(pred.payload, ContinuousForecast):
            print(f"  point=${pred.payload.point_forecast:.2f}")

print(f"\n✓ All {NUM_RUNS} runs complete")


RUN 1 / 3


/home/coder/agentic-forecasting/.venv/lib/python3.12/site-packages/google/adk/tools/function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(


  point=$89.95
  point=$90.33
  point=$88.26

RUN 2 / 3
  point=$88.10
  point=$87.78
  point=$88.26

RUN 3 / 3

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

  point=$88.10
  point=$87.78
  point=$88.26

✓ All 3 runs complete


## LLM Actions and Historical Governor Comparison

For each run and horizon: the LLM's proposed rank and uncertainty action, what evidence
tier the policy actually granted (a rank can get clipped down if evidence is thin), and
the historical delta percentiles that governed the shift. The `historical_delta_*`
values should be identical across runs at a given horizon — they come from price
history, not the LLM — any drift there points at a real bug, not LLM variance.

In [3]:
RANK_LABEL = {-2: "strong bearish (p10)", -1: "mild bearish (p25)", 0: "neutral (p50)",
              1: "mild bullish (p75)", 2: "strong bullish (p90)"}

for run_data in results:
    run_num = run_data["run_num"]
    predictions = run_data["predictions"]

    print(f"\n{'─'*72}")
    print(f"RUN {run_num}")
    print(f"{'─'*72}")

    assessment = None
    for pred in predictions:
        ft = pred.metadata.get("forecast_transformation")
        pd_decision = pred.metadata.get("policy_decision")
        if assessment is None:
            assessment = pred.metadata.get("llm_context_assessment")
        if not ft or not pd_decision:
            continue

        horizon = ft["horizon"]
        llm_rank = None
        if assessment:
            for action in assessment.get("horizon_actions", []):
                if action["horizon"] == horizon:
                    llm_rank = action["center_action"]
                    break

        print(f"\n  h={horizon}d:")
        print(f"    LLM proposed rank:      {llm_rank} ({RANK_LABEL.get(llm_rank, '?')})")
        print(f"    Policy-granted rank:    {pd_decision['center_action']} (evidence_tier={pd_decision['evidence_tier']})")
        print(f"    Uncertainty action:     {pd_decision['uncertainty_action']}")
        print(f"    Historical delta p10/p50/p90: {ft['historical_delta_p10']:.2f} / {ft['historical_delta_p50']:.2f} / {ft['historical_delta_p90']:.2f}")
        print(f"    Applied center shift:   {ft['applied_center_adjustment']:+.2f}")
        print(f"    Final point forecast:   ${ft['final_point_forecast']:.2f}  (ensemble baseline was ${ft['original_point_forecast']:.2f})")

    if assessment:
        print(f"\n  Confidence: {assessment.get('confidence')}")
        print(f"  Overall rationale: {assessment.get('overall_rationale')}")


────────────────────────────────────────────────────────────────────────
RUN 1
────────────────────────────────────────────────────────────────────────

  h=5d:
    LLM proposed rank:      1 (mild bullish (p75))
    Policy-granted rank:    1 (evidence_tier=corroborated)
    Uncertainty action:     moderately_wider
    Historical delta p10/p50/p90: -4.06 / 0.26 / 3.94
    Applied center shift:   +1.85
    Final point forecast:   $89.95  (ensemble baseline was $88.10)

  h=10d:
    LLM proposed rank:      1 (mild bullish (p75))
    Policy-granted rank:    1 (evidence_tier=corroborated)
    Uncertainty action:     moderately_wider
    Historical delta p10/p50/p90: -5.86 / 0.48 / 5.59
    Applied center shift:   +2.55
    Final point forecast:   $90.33  (ensemble baseline was $87.78)

  h=21d:
    LLM proposed rank:      0 (neutral (p50))
    Policy-granted rank:    0 (evidence_tier=limited)
    Uncertainty action:     small_wider
    Historical delta p10/p50/p90: -8.44 / 0.71 / 8.29
    

## Compare Point Forecasts Across Runs

In [4]:
forecast_comparison = {h: [] for h in HORIZONS}

for run_data in results:
    for pred in run_data["predictions"]:
        if isinstance(pred.payload, ContinuousForecast):
            as_of = pd.Timestamp(pred.as_of)
            forecast_date = pd.Timestamp(pred.forecast_date)
            offset = pd.tseries.offsets.BDay()
            for h in HORIZONS:
                if (as_of + offset * h).normalize() == forecast_date.normalize():
                    forecast_comparison[h].append(pred.payload.point_forecast)
                    break

comparison_rows = []
for h in HORIZONS:
    forecasts = forecast_comparison[h]
    if forecasts:
        comparison_rows.append({
            "Horizon": f"{h}d",
            "Run 1": f"${forecasts[0]:.2f}" if len(forecasts) > 0 else "—",
            "Run 2": f"${forecasts[1]:.2f}" if len(forecasts) > 1 else "—",
            "Run 3": f"${forecasts[2]:.2f}" if len(forecasts) > 2 else "—",
            "Mean": f"${np.mean(forecasts):.2f}",
            "Std Dev": f"${np.std(forecasts):.2f}",
            "Range": f"${np.max(forecasts) - np.min(forecasts):.2f}",
        })

df_comparison = pd.DataFrame(comparison_rows)
print("\n" + "="*72)
print("POINT FORECAST COMPARISON ACROSS 3 RUNS")
print("="*72)
print(df_comparison.to_string(index=False))


POINT FORECAST COMPARISON ACROSS 3 RUNS
Horizon  Run 1  Run 2  Run 3   Mean Std Dev Range
     5d $89.95 $88.10 $88.10 $88.72   $0.87 $1.85
    10d $90.33 $87.78 $87.78 $88.63   $1.20 $2.55
    21d $88.26 $88.26 $88.26 $88.26   $0.00 $0.00


## Extract Full Distributions (Quantiles)

In [5]:
all_distributions = {run_idx: {h: None for h in HORIZONS} for run_idx in range(NUM_RUNS)}

for run_idx, run_data in enumerate(results):
    for pred in run_data["predictions"]:
        if isinstance(pred.payload, ContinuousForecast):
            cf = pred.payload
            as_of = pd.Timestamp(pred.as_of)
            forecast_date = pd.Timestamp(pred.forecast_date)
            offset = pd.tseries.offsets.BDay()
            for h in HORIZONS:
                if (as_of + offset * h).normalize() == forecast_date.normalize():
                    if all_distributions[run_idx][h] is None:
                        all_distributions[run_idx][h] = cf
                    break

dist_rows = []
for h in HORIZONS:
    row = {"Horizon": f"{h}d"}
    for run_idx in range(NUM_RUNS):
        cf = all_distributions[run_idx][h]
        if cf is not None:
            point = f"${cf.point_forecast:.2f}"
            ci = ""
            if cf.quantiles:
                q10 = cf.quantiles.get(0.1)
                q90 = cf.quantiles.get(0.9)
                if q10 is not None and q90 is not None:
                    ci = f" [{q10:.2f}, {q90:.2f}]"
            row[f"Run {run_idx+1}"] = point + ci
    dist_rows.append(row)

df_distributions = pd.DataFrame(dist_rows)
print("\n" + "="*72)
print("FULL DISTRIBUTIONS BY HORIZON (point + 80% CI)")
print("="*72)
print(df_distributions.to_string(index=False))


FULL DISTRIBUTIONS BY HORIZON (point + 80% CI)
Horizon                 Run 1                 Run 2                 Run 3
     5d $89.95 [84.97, 94.57] $88.10 [82.38, 93.40] $88.10 [82.38, 93.40]
    10d $90.33 [83.75, 97.49] $87.78 [79.76, 96.51] $87.78 [79.76, 96.51]
    21d $88.26 [78.31, 96.71] $88.26 [75.63, 98.99] $88.26 [75.63, 98.99]


## Extract Rationales

In [6]:
for run_idx, run_data in enumerate(results):
    print(f"\n{'='*72}")
    print(f"RUN {run_idx + 1} — OVERALL RATIONALE / RESEARCH SUMMARY")
    print(f"{'='*72}\n")

    predictions = run_data["predictions"]
    seen = set()
    for pred in predictions:
        assessment = pred.metadata.get("llm_context_assessment") if pred.metadata else None
        if not assessment:
            continue
        text = f"[confidence={assessment.get('confidence')}] {assessment.get('overall_rationale')}"
        if text not in seen:
            print(text)
            print()
            seen.add(text)


RUN 1 — OVERALL RATIONALE / RESEARCH SUMMARY

[confidence=0.9] The combination of persistent Strait of Hormuz transit disruptions, reduced global supply outlooks from the IEA, and a market structure prioritizing immediate supply (backwardation) provides a bullish case for WTI in the near term. The supply constraints are largely new relative to historical model data, supporting elevated confidence in a directional risk premium.


RUN 2 — OVERALL RATIONALE / RESEARCH SUMMARY

[confidence=0.85] Evidence confirms a state of partial supply disruption and significant geopolitical risk, providing upward price support. The market structure of backwardation and the persistence of risk premia justify a bullish near-term stance. However, the projected gradual increase in OPEC+ production and potential market assessment of transit workarounds introduce uncertainty for longer-term price stability.


RUN 3 — OVERALL RATIONALE / RESEARCH SUMMARY

[confidence=0.0] Python applied the audited neutral f

## Consistency Assessment

In [7]:
print("\n" + "="*72)
print("CONSISTENCY METRICS")
print("="*72)

for h in HORIZONS:
    forecasts = forecast_comparison[h]
    if len(forecasts) == NUM_RUNS:
        mean = np.mean(forecasts)
        std = np.std(forecasts)
        cv = (std / mean) * 100
        print(f"\nh={h}d:")
        print(f"  Mean:                 ${mean:.2f}")
        print(f"  Std Dev:              ${std:.2f}")
        print(f"  Coefficient of Var:   {cv:.1f}%")
        print(f"  Range (max - min):    ${np.max(forecasts) - np.min(forecasts):.2f}")

        if cv < 1.0:
            consistency = "✓ Very consistent (CV < 1%)"
        elif cv < 2.0:
            consistency = "✓ Consistent (CV 1-2%)"
        elif cv < 5.0:
            consistency = "⚠ Moderate variance (CV 2-5%)"
        else:
            consistency = "✗ High variance (CV > 5%)"
        print(f"  Assessment:           {consistency}")


CONSISTENCY METRICS

h=5d:
  Mean:                 $88.72
  Std Dev:              $0.87
  Coefficient of Var:   1.0%
  Range (max - min):    $1.85
  Assessment:           ✓ Very consistent (CV < 1%)

h=10d:
  Mean:                 $88.63
  Std Dev:              $1.20
  Coefficient of Var:   1.4%
  Range (max - min):    $2.55
  Assessment:           ✓ Consistent (CV 1-2%)

h=21d:
  Mean:                 $88.26
  Std Dev:              $0.00
  Coefficient of Var:   0.0%
  Range (max - min):    $0.00
  Assessment:           ✓ Very consistent (CV < 1%)
